# LPKT H2

Este notebook foi gerado automaticamente para executar o experimento **LPKT H2**.

## Objetivo
Este notebook serve para:

1. configurar o ambiente (local ou Colab),
2. localizar corretamente a raiz do projeto,
3. garantir acesso ao código em `src/`,
4. opcionalmente montar o Google Drive e copiar dados,
5. executar o treino do modelo,
6. extrair histórico de treino,
7. plotar gráficos de **Loss** e **AUC**,
8. salvar resumos em `artifacts/colab_summaries/`.


## Etapa 1 — Detectar ambiente e localizar a raiz do projeto

Nesta etapa, o notebook identifica se está rodando no **Google Colab** ou no **ambiente local**.

### O que esta célula resolve
- no Colab: clona automaticamente o repositório, instala dependências e usa `/content/ai-core`
- no local: detecta corretamente a raiz do projeto mesmo se o notebook estiver sendo executado de dentro de `notebooks/colab/`
- adiciona `src/` ao `sys.path`, evitando erros como `ModuleNotFoundError: brain_kt`


In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/GuilhermeDesoler/ai-core.git'

if IN_COLAB:
    REPO_DIR = Path('/content/ai-core')
    if not REPO_DIR.exists():
        get_ipython().system(f'git clone -b improve/high-impact-training {REPO_URL} /content/ai-core')
    get_ipython().run_line_magic('cd', '/content/ai-core')
    get_ipython().system('pip install -q -r requirements.txt')
else:
    cwd = Path.cwd().resolve()

    # Caso 1: notebook aberto em ai-core/notebooks/colab
    if cwd.name == 'colab' and cwd.parent.name == 'notebooks':
        REPO_DIR = cwd.parents[1]
    # Caso 2: notebook aberto em ai-core/notebooks
    elif cwd.name == 'notebooks':
        REPO_DIR = cwd.parent
    # Caso 3: notebook aberto já na raiz ai-core
    else:
        REPO_DIR = cwd

SRC_PATH = REPO_DIR / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('IN_COLAB:', IN_COLAB)
print('CWD:', Path.cwd().resolve())
print('REPO_DIR:', REPO_DIR)
print('SRC_PATH:', SRC_PATH)
print('scripts exists:', (REPO_DIR / 'scripts').exists())
print('src exists:', SRC_PATH.exists())


## Etapa 2 — Dados e integração com Google Drive

Esta etapa prepara o acesso aos dados.

### Comportamento esperado
- no Colab: monta o Google Drive e permite copiar dados de `MyDrive/data/`
- no local: apenas valida se os artefatos esperados já existem em `data/`

### Estrutura esperada dos dados no Drive
```text
MyDrive/data/
  raw/
  processed/
```

### Observação
As linhas de cópia ficam comentadas por padrão para evitar operações desnecessárias.
Descomente apenas se quiser copiar os dados do Drive para dentro do diretório do projeto no Colab.


In [ ]:
from pathlib import Path
import shutil

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive mount opcional:', e)

PROJECT_ROOT = REPO_DIR
DATA_TARGET = PROJECT_ROOT / 'data'
DATA_TARGET.mkdir(parents=True, exist_ok=True)

SOURCE_RAW_DIR = Path('/content/drive/MyDrive/data/raw')
SOURCE_PROCESSED_DIR = Path('/content/drive/MyDrive/data/processed')

# Descomente se quiser copiar do Drive no Colab:
# shutil.copytree(SOURCE_RAW_DIR, DATA_TARGET / 'raw', dirs_exist_ok=True)
# shutil.copytree(SOURCE_PROCESSED_DIR, DATA_TARGET / 'processed', dirs_exist_ok=True)

for p in [
    REPO_DIR / 'data' / 'raw' / 'answers.json',
    REPO_DIR / 'data' / 'processed' / 'sequences' / 'user_sequences.json',
]:
    print(p, p.exists())


## Etapa 3 — Regenerar artefatos base (opcional)

Use esta etapa se você ainda não possui os artefatos processados ou se quiser reconstruir tudo.

### O que pode ser regenerado
- `answers_prepared.csv`
- `user_sequences.json`
- mapeamentos H2/H3
- sequências por sessão

Por padrão, esta célula fica comentada para evitar reprocessamentos desnecessários.


In [ ]:
# Opcional: regenere os artefatos base e sequências por sessão.
# get_ipython().system('python run_data_pipeline.py')
# get_ipython().system('python scripts/build_user_session_sequences.py')


## Etapa 4 — Executar o treino do experimento

Nesta etapa, o notebook roda o script oficial do experimento correspondente.

### O que acontece aqui
- o script é executado dentro da raiz do projeto,
- o `PYTHONPATH` é ajustado para usar `src/`,
- o stdout do treino é capturado,
- o notebook tenta extrair automaticamente o histórico de epochs a partir dos logs impressos.


In [ ]:
import subprocess
import re
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

SCRIPT_PATH = 'scripts/train_eval_lpkt_topic_h2.py'
SUMMARY_NAME = 'lpkt_h2'

cmd = f'PYTHONPATH=./src python scripts/train_eval_lpkt_topic_h2.py'
print('Running:', cmd)

result = subprocess.run(
    cmd,
    shell=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Command failed with code {result.returncode}')

pattern = re.compile(
    r'Epoch\s+(\d+)(?:/(\d+))?\s*\|\s*'
    r'Train Loss:\s*([0-9.]+)'
    r'(?:\s*\|\s*Train AUC:\s*([0-9.]+))?'
    r'\s*\|\s*Val Loss:\s*([0-9.]+)'
    r'\s*\|\s*Val AUC:\s*([0-9.]+)'
    r'\s*\|\s*Val Acc:\s*([0-9.]+)'
    r'\s*\|\s*LR:\s*([0-9.eE+-]+)'
)

rows = []
for line in result.stdout.splitlines():
    m = pattern.search(line)
    if m:
        rows.append(
            {
                'epoch': int(m.group(1)),
                'train_loss': float(m.group(3)),
                'train_auc': float(m.group(4)) if m.group(4) else None,
                'val_loss': float(m.group(5)),
                'val_auc': float(m.group(6)),
                'val_acc': float(m.group(7)),
                'lr': float(m.group(8)),
            }
        )

history_df = pd.DataFrame(rows)
history_df


## Etapa 5 — Visualizar curvas de Loss e AUC

Esses gráficos servem para diagnosticar o comportamento do treino.

### O que observar
- **Train Loss ↓ e Val Loss ↑**: possível overfitting
- **Val AUC subindo**: o modelo está generalizando melhor
- **Val AUC estabilizada/caindo**: o modelo pode ter atingido o limite ou começado a overfitar


In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.plot(history_df['epoch'], history_df['train_loss'], label='Train Loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='Val Loss')
plt.title(SUMMARY_NAME + ' - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

if history_df['train_auc'].notna().any():
    fig = plt.figure(figsize=(10, 5))
    plt.plot(history_df['epoch'], history_df['train_auc'], label='Train AUC')
    plt.plot(history_df['epoch'], history_df['val_auc'], label='Val AUC')
    plt.title(SUMMARY_NAME + ' - AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


## Etapa 6 — Ler métricas finais e salvar resumo consolidado

Nesta etapa, o notebook tenta localizar o diretório do run salvo pelo script oficial,
carrega o `metrics.json` e salva um resumo padronizado em:

```text
artifacts/colab_summaries/
```

Isso permite comparar todos os experimentos depois em um único notebook.


In [ ]:
saved_match = re.search(r'Saved run to\s+(.+)', result.stdout)
run_dir = Path(saved_match.group(1).strip()) if saved_match else None

metrics = {}
if run_dir and (run_dir / 'metrics.json').exists():
    metrics = json.loads((run_dir / 'metrics.json').read_text())

metrics

summary_dir = Path('artifacts/colab_summaries')
summary_dir.mkdir(parents=True, exist_ok=True)

(summary_dir / f'lpkt_h2_history.csv').write_text(history_df.to_csv(index=False))
(summary_dir / f'lpkt_h2_metrics.json').write_text(json.dumps(metrics, indent=2))

print('Saved summary files to', summary_dir)
